# Gene-grouped cross-validation (XGBoost)

Uses `src/split_train_valid_test.py` to assign each gene to one of `n_folds` CV folds,
then trains XGBoost once per fold with **hyperparameter tuning on a valid fold**.

Fold roles for test fold `k` (deterministic rotation, not a random subsample):

| role | which fold |
|------|------------|
| test | `k` |
| valid | `(k + 1) % n_folds` — used to pick hyperparameters |
| train | all remaining folds |

Genes never appear in more than one role within a round. Reported metrics are always on **test**.


In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.model_selection import ParameterSampler
from xgboost import XGBClassifier


In [ ]:
load_dotenv()
project_root = Path(os.environ["PROJECT_ROOT"])
sys.path.insert(0, str(project_root / "src"))

from split_train_valid_test import (
    FEATURE_COLUMNS,
    GROUP_COLUMN,
    align_categories_to_train,
    build_feature_matrix,
    iter_cv_folds,
    run_cv_folds,
    split_summary,
)

N_FOLDS = 5
SEED = 42
N_TUNING_TRIALS = 12  # random hyperparameter configs scored on valid each fold
MODELS_DIR = project_root / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

PARAM_DISTRIBUTIONS = {
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_child_weight": [1, 5, 10],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "reg_lambda": [0.5, 1.0, 5.0],
}


## Assign CV folds

Calls `run_cv_folds()` (same as `python src/split_train_valid_test.py --mode cv`).
Writes `data/processed/cv_folds.parquet`.

In [ ]:
folded = run_cv_folds(n_folds=N_FOLDS, seed=SEED)
summary = split_summary(folded, partition_col="fold")
display(summary)

gene_folds = folded.groupby(GROUP_COLUMN)["fold"].nunique()
assert (gene_folds == 1).all()
print(f"{folded[GROUP_COLUMN].nunique()} genes, {len(folded)} variants, gene overlap across folds: none")

## Cross-validated XGBoost + valid-fold hyperparameter tuning

For each outer fold: sample `N_TUNING_TRIALS` hyperparameter configs, fit on train with
early stopping on valid, keep the config with best **valid ROC-AUC**, then score that
model on the held-out **test** fold.


In [ ]:
def make_model(**params) -> XGBClassifier:
    return XGBClassifier(
        n_estimators=500,
        objective="binary:logistic",
        eval_metric="auc",
        enable_categorical=True,
        tree_method="hist",
        random_state=SEED,
        n_jobs=4,
        early_stopping_rounds=50,
        **params,
    )


def tune_on_valid(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_valid: pd.DataFrame,
    y_valid: pd.Series,
    *,
    n_trials: int,
    seed: int,
) -> tuple[XGBClassifier, dict, float, pd.DataFrame]:
    """Search hyperparameters by valid ROC-AUC; return best fitted model."""
    sampler = ParameterSampler(PARAM_DISTRIBUTIONS, n_iter=n_trials, random_state=seed)
    trial_rows = []
    best_model = None
    best_params = None
    best_valid_auc = -np.inf

    for trial_idx, params in enumerate(sampler):
        model = make_model(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=False,
        )
        proba_valid = model.predict_proba(X_valid)[:, 1]
        valid_auc = roc_auc_score(y_valid, proba_valid)
        trial_rows.append(
            {
                "trial": trial_idx,
                "valid_roc_auc": valid_auc,
                "best_iteration": model.best_iteration,
                **params,
            }
        )
        if valid_auc > best_valid_auc:
            best_valid_auc = valid_auc
            best_model = model
            best_params = dict(params)

    return best_model, best_params, best_valid_auc, pd.DataFrame(trial_rows)


fold_rows = []
oof_parts = []
importances = []
tuning_logs = []

for fold_data in iter_cv_folds(folded, n_folds=N_FOLDS, valid_fold_offset=1):
    k = fold_data["fold"]
    train_df, valid_df, test_df = fold_data["train"], fold_data["valid"], fold_data["test"]

    X_train, y_train = build_feature_matrix(train_df)
    X_valid, y_valid = build_feature_matrix(valid_df)
    X_test, y_test = build_feature_matrix(test_df)
    X_train, X_valid, X_test = align_categories_to_train(X_train, X_valid, X_test)

    model, best_params, best_valid_auc, trials = tune_on_valid(
        X_train,
        y_train,
        X_valid,
        y_valid,
        n_trials=N_TUNING_TRIALS,
        seed=SEED + k,
    )
    trials.insert(0, "fold", k)
    tuning_logs.append(trials)

    proba_test = model.predict_proba(X_test)[:, 1]
    row = {
        "fold": k,
        "valid_fold": fold_data["valid_fold"],
        "n_train": len(train_df),
        "n_valid": len(valid_df),
        "n_test": len(test_df),
        "n_test_genes": test_df[GROUP_COLUMN].nunique(),
        "best_valid_roc_auc": best_valid_auc,
        "best_iteration": model.best_iteration,
        "roc_auc": roc_auc_score(y_test, proba_test),
        "pr_auc": average_precision_score(y_test, proba_test),
        **{f"param_{key}": value for key, value in best_params.items()},
    }
    fold_rows.append(row)
    print(
        f"fold {k}: valid AUC={best_valid_auc:.3f} → test ROC-AUC={row['roc_auc']:.3f}  "
        f"PR-AUC={row['pr_auc']:.3f}  best={best_params}"
    )

    oof_parts.append(
        pd.DataFrame(
            {
                "VariationID": test_df["VariationID"].values,
                "gene": test_df[GROUP_COLUMN].values,
                "fold": k,
                "y_true": y_test.values,
                "y_proba": proba_test,
            }
        )
    )
    importances.append(
        pd.Series(model.feature_importances_, index=FEATURE_COLUMNS, name=f"fold_{k}")
    )

fold_metrics = pd.DataFrame(fold_rows).set_index("fold")
oof = pd.concat(oof_parts, ignore_index=True)
tuning_log = pd.concat(tuning_logs, ignore_index=True)
fold_metrics


## Chosen hyperparameters per fold

Each fold picks the config with the highest valid ROC-AUC. Test metrics above are from that chosen model only.


In [ ]:
param_cols = [c for c in fold_metrics.columns if c.startswith("param_")]
display(fold_metrics[["valid_fold", "best_valid_roc_auc", "roc_auc", "pr_auc"] + param_cols].round(3))

print("All tuning trials (one row per config × fold):")
tuning_log.sort_values(["fold", "valid_roc_auc"], ascending=[True, False]).groupby("fold").head(3)


## CV summary

In [ ]:
summary_stats = fold_metrics[["roc_auc", "pr_auc"]].agg(["mean", "std", "min", "max"])
oof_roc = roc_auc_score(oof["y_true"], oof["y_proba"])
oof_pr = average_precision_score(oof["y_true"], oof["y_proba"])

print(f"Out-of-fold ROC-AUC={oof_roc:.3f}  PR-AUC={oof_pr:.3f}  (n={len(oof)})")
summary_stats.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for fold, part in oof.groupby("fold"):
    fpr, tpr, _ = roc_curve(part["y_true"], part["y_proba"])
    axes[0].plot(
        fpr,
        tpr,
        alpha=0.7,
        label=f"fold {fold} ({roc_auc_score(part['y_true'], part['y_proba']):.3f})",
    )
    prec, rec, _ = precision_recall_curve(part["y_true"], part["y_proba"])
    axes[1].plot(
        rec,
        prec,
        alpha=0.7,
        label=f"fold {fold} ({average_precision_score(part['y_true'], part['y_proba']):.3f})",
    )

fpr, tpr, _ = roc_curve(oof["y_true"], oof["y_proba"])
axes[0].plot(fpr, tpr, c="black", lw=2, label=f"OOF ({oof_roc:.3f})")
axes[0].plot([0, 1], [0, 1], ls="--", c="gray", lw=1)
axes[0].set_xlabel("FPR")
axes[0].set_ylabel("TPR")
axes[0].set_title("ROC by fold")
axes[0].legend(fontsize=8)

prec, rec, _ = precision_recall_curve(oof["y_true"], oof["y_proba"])
axes[1].plot(rec, prec, c="black", lw=2, label=f"OOF ({oof_pr:.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision–recall by fold")
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()

## Feature importance across folds

In [ ]:
importance_df = pd.concat(importances, axis=1)
importance_df["mean"] = importance_df.mean(axis=1)
importance_df["std"] = importance_df.std(axis=1)
importance_df = importance_df.sort_values("mean", ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(
    importance_df.index[::-1],
    importance_df["mean"].iloc[::-1],
    xerr=importance_df["std"].iloc[::-1],
    color="#3b6ea5",
    alpha=0.9,
)
ax.set_xlabel("Mean gain ± std across folds")
ax.set_title("XGBoost feature importance (CV)")
fig.tight_layout()
plt.show()

importance_df[["mean", "std"]].round(4)

## Persist OOF predictions

In [ ]:
oof_path = MODELS_DIR / "xgb_cv_oof_predictions.parquet"
metrics_path = MODELS_DIR / "xgb_cv_fold_metrics.csv"
tuning_path = MODELS_DIR / "xgb_cv_tuning_log.csv"
oof.to_parquet(oof_path, index=False)
fold_metrics.to_csv(metrics_path)
tuning_log.to_csv(tuning_path, index=False)
print("Saved", oof_path)
print("Saved", metrics_path)
print("Saved", tuning_path)
fold_metrics.round(3)
